# Spherical datasets in autocast

This notebook loads one shard of the spherical shallow-water datasets of
[aiphys#221](https://github.com/alan-turing-institute/aiphys/issues/221), published as
[`alan-turing-aiphys/aiphys-datasets`](https://huggingface.co/datasets/alan-turing-aiphys/aiphys-datasets),
and passes it through autocast's identity encoder and decoder.

Every shard is a dictionary saved with `torch.save`. Next to the fields it holds `grid_type`, the
torch-harmonics name of the grid of those fields (`"legendre-gauss"` for every dataset of the
benchmark). `SpatioTemporalDataset` maps the file from disk, reads one window of frames per item,
and returns spherical samples that keep the name of their grid through collation, device transfer
and the identity encoder.

The notebook runs on a Colab CPU runtime, with the shard `ds5_era5_weather/train/batch_0000.pt`
(8 runs of 41 frames on the 384 x 768 Gauss-Legendre grid, 1.2 GB). Colab runs Python 3.13, and
autocast and autosim declare Python below 3.13, hence `--ignore-requires-python`. autocast takes
autosim and autouq from GitHub through uv, which pip does not read, so the first install line puts
them in place: autosim from the branch whose spherical simulator writes `grid_type`, and autouq at
the commit pinned by autocast.

In [ ]:
AUTOCAST = (
    "git+https://github.com/alan-turing-institute/autocast@load-spherical-datasets"
)
AUTOSIM = "git+https://github.com/alan-turing-institute/autosim@spherical-swe-galewsky"
AUTOUQ = "git+https://github.com/alan-turing-institute/autouq@3d3626f149bdfc76161bbb0075f419a9b83fd7a2"
FLAGS = "-q --ignore-requires-python"

%pip install {FLAGS} "autouq @ {AUTOUQ}" "autosim @ {AUTOSIM}"
%pip install {FLAGS} "autocast @ {AUTOCAST}"

The Hugging Face Hub appears only in the next cell: `hf download` puts the shard and its JSON
sidecar in the local cache. autosim and autocast never contact the Hub; they read a local path.

In [ ]:
REPO = "alan-turing-aiphys/aiphys-datasets"
PATTERN = "ds5_era5_weather/train/batch_0000.*"

!hf download {REPO} --repo-type dataset --include "{PATTERN}"

The shard is read from the cache with `local_files_only=True`. `torch.load(mmap=True)` maps the
file, so the memory of the process barely changes until a tensor is read. The payload holds
`grid_type` as a plain string, which `torch.load` accepts with its default `weights_only=True`.

In [ ]:
import json
from pathlib import Path

import psutil
import torch
from huggingface_hub import snapshot_download


def rss_gb() -> float:
    """Resident memory of this process, in GB."""
    return psutil.Process().memory_info().rss / 1e9


root = Path(
    snapshot_download(
        REPO, repo_type="dataset", local_files_only=True, allow_patterns=[PATTERN]
    )
)
shard = root / "ds5_era5_weather/train/batch_0000.pt"
sidecar = json.loads(shard.with_suffix(".json").read_text())

size_gb = shard.stat().st_size / 1e9
ram_gb = psutil.virtual_memory().total / 1e9
print(f"shard {size_gb:.2f} GB, machine RAM {ram_gb:.1f} GB")
print(f"RSS before loading: {rss_gb():.2f} GB")
payload = torch.load(shard, mmap=True)
print(f"RSS after torch.load(mmap=True): {rss_gb():.2f} GB")
for key, value in payload.items():
    print(f"  {key}: {tuple(value.shape) if torch.is_tensor(value) else repr(value)}")
print("sidecar grid_type:", sidecar["grid_type"])
print("sampled parameters:", sidecar["param_names"])

`SpatioTemporalDataset` maps the same file. Construction reads the shapes only; item `i` reads
the frames of its own window, here one input frame and four output frames of one run, and converts
them to `float32`. With a grid on the sphere in the payload, the items are `SphericalSample`s.

In [ ]:
from autocast.data.dataset import SpatioTemporalDataset

dataset = SpatioTemporalDataset(data_path=str(shard), n_steps_input=1, n_steps_output=4)
print(f"{len(dataset)} windows, {dataset.n_trajectories} runs, {dataset.grid_type!r}")
print(f"RSS after building the dataset: {rss_gb():.2f} GB")

sample = dataset[0]
print(type(sample).__name__, sample.grid_type)
print("input", tuple(sample.input_fields.shape))
print("output", tuple(sample.output_fields.shape))
print("constant fields", tuple(sample.constant_fields.shape))
print("constant scalars", tuple(sample.constant_scalars.shape))

`collate_batches` stacks spherical samples into a `SphericalBatch` on their grid. Two worker
processes share the mapping of the file. `to` and `pin_memory`, which Lightning and the `DataLoader`
call on every batch, return a `SphericalBatch` on the same grid.

In [ ]:
from torch.utils.data import DataLoader

from autocast.types import collate_batches

loader = DataLoader(
    dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_batches
)
batch = next(iter(loader))
print(type(batch).__name__, batch.grid_type)
print("input_fields", tuple(batch.input_fields.shape))
print("output_fields", tuple(batch.output_fields.shape))
moved = batch.to("cpu")
print("after to('cpu'):", type(moved).__name__, moved.grid_type)
print(f"RSS after one batch: {rss_gb():.2f} GB")

The identity encoder keeps the fields on their grid, so a `SphericalBatch` gives an
`EncodedSphericalBatch` on the same grid, with the sampled parameters as global conditioning. The
identity decoder returns its input, so the decoded fields equal the fields of the batch.

The figure shows the height `h` of the first run of the batch: its input frame, and its last output
frame after encoding and decoding. On the Gauss-Legendre grid the latitudes are the arcsines of the
Gauss-Legendre nodes, and the rows of the data run from north to south.

In [ ]:
from autocast.decoders.identity import IdentityDecoder
from autocast.encoders.identity import IdentityEncoder

n_channels = batch.input_fields.shape[-1]
encoder = IdentityEncoder(in_channels=n_channels)
decoder = IdentityDecoder(in_channels=encoder.latent_channels)

encoded = encoder.encode_batch(batch)
decoded = decoder.decode(encoded.encoded_output_fields)
print(type(encoded).__name__, encoded.grid_type)
print("encoded_inputs", tuple(encoded.encoded_inputs.shape))
print("global_cond", tuple(encoded.global_cond.shape))
print("decoded equals batch.output_fields:", torch.equal(decoded, batch.output_fields))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_lat, n_lon = batch.input_fields.shape[2:4]
assert batch.grid_type == "legendre-gauss"
nodes, _ = np.polynomial.legendre.leggauss(n_lat)
latitudes = np.degrees(np.arcsin(nodes))[::-1]
longitudes = np.arange(n_lon) * 360.0 / n_lon

frames = {
    "input frame": batch.input_fields[0, 0, ..., 0],
    "decoded last output frame": decoded[0, -1, ..., 0],
}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), constrained_layout=True)
for ax, (title, field) in zip(axes, frames.items(), strict=True):
    mesh = ax.pcolormesh(
        longitudes, latitudes, field.numpy(), cmap="RdBu_r", shading="nearest"
    )
    ax.set_title(f"h [m], {title}, {batch.grid_type}")
    ax.set_xlabel("longitude [deg]")
    ax.set_ylabel("latitude [deg]")
    fig.colorbar(mesh, ax=ax)
fig.savefig("/content/spherical_batch.png", dpi=100)
plt.show()